In [29]:
from pathlib import Path
notebook_directory = Path.cwd()

import numpy as np
import pandas as pd
import xlwings as xw

In [30]:
sorted_symbols = ['FSTA', 'VDC']

prices_csv = 'sectors'

start_date = pd.Timestamp("2026-01-01")
end_date   = pd.Timestamp("2026-08-15")

output_filename = "_".join(sorted_symbols) + ".csv"

# ishare_fee_diff = .003

In [31]:
prices_path = (
    notebook_directory.parent #.parent
    / "backtesting"
    / "historical prices"
    / f"{prices_csv}.csv"
)
prices_df = pd.read_csv(prices_path, parse_dates=["date"])


db_path = (
    notebook_directory.parent
    / "trading"
    / "spreadsheets"
    / "2026 Fin Inst Database.xlsx"
)
wb = xw.Book(db_path)
ws = wb.sheets["Scalar Inputs Table"]
db_df = ws.tables["scalar_inputs_table"].range.options(pd.DataFrame, header=1, index=False).value



scalar_path = (
    notebook_directory.parent
    / "trading"
    / "spreadsheets"
    / "2026 Group Trading Inputs.xlsm"
)
wb = xw.Book(scalar_path)
ws = wb.sheets["ANCHOR INPUTS"]
scalar_df = ws.tables["anchor_inputs"].range.options(pd.DataFrame, header=1, index=False).value

In [32]:
scalar_df = scalar_df[scalar_df["my_fi_name"].isin(sorted_symbols)].copy()

In [33]:
#previous_month = (pd.Timestamp.today() - pd.DateOffset(months=1)).to_period("M")
#prices_df = prices_df.loc[prices_df["date"].dt.to_period("M").eq(previous_month)].copy()

prices_df["date"] = pd.to_datetime(prices_df["date"], errors="coerce")
prices_df = prices_df.loc[prices_df["date"].between(start_date, end_date)].copy()

df = prices_df[['date'] + sorted_symbols].copy()

In [34]:
non_anchor = sorted_symbols[0]
anchor = sorted_symbols[1]
ratio_name = f'{anchor}/{non_anchor}'

df[ratio_name] = df[anchor] / df[non_anchor]

In [35]:
df[f'{ratio_name} avg'] = df[ratio_name].mean()
df[f"{ratio_name} 10 dma"] = (df[ratio_name].rolling(window=10, min_periods=10).mean())
df[f"{ratio_name} 20 dma"] = (df[ratio_name].rolling(window=20, min_periods=20).mean())

df[f'{non_anchor} avg'] = df[non_anchor] * df[f'{ratio_name} avg']
df[f'{non_anchor} 10 dma'] = df[non_anchor] * df[f'{ratio_name} 10 dma']
df[f'{non_anchor} 20 dma'] = df[non_anchor] * df[f'{ratio_name} 20 dma']

df[f'avg diff'] = np.log(df[anchor] / df[f'{non_anchor} avg'])
df[f'10 dma diff'] = np.log(df[anchor] / df[f'{non_anchor} 10 dma'])
df[f'20 dma diff'] = np.log(df[anchor] / df[f'{non_anchor} 20 dma'])

In [38]:
for sym in sorted_symbols:
    df[f"{sym} div"] = 0.0

    rows = db_df.loc[db_df["symbol"] == sym]
    if rows.empty:
        continue

    row = rows.iloc[0]
    print(row)

    for month in range(1,9,1):
        core = "div 2026-0" + str(month)
        amt = float(row[core])
        if amt != 0:
            ex_date = pd.to_datetime(row[f"{core} ex-date"])
            df.loc[df['date'] == ex_date, f"{sym} div"] = amt

group                                   Sector
sector                        Consumer Staples
symbol                                    FSTA
etf family                            Fidelity
etf fee                                 0.0008
90-day avg daily volume               121983.0
div 2026-08                                  0
div 2026-08 ex-date                          -
div 2026-07                                  0
div 2026-07 ex-date                          -
div 2026-06                              0.296
div 2026-06 ex-date        2026-06-18 00:00:00
div 2026-05                                  0
div 2026-05 ex-date                          -
div 2026-04                                  0
div 2026-04 ex-date                          -
div 2026-03                               0.29
div 2026-03 ex-date        2026-03-20 00:00:00
div 2026-02                                  0
div 2026-02 ex-date                          -
div 2026-01                                  0
div 2026-01 e

In [39]:
print(df)

           date   FSTA     VDC  VDC/FSTA  VDC/FSTA avg  VDC/FSTA 10 dma  \
1090 2026-01-02  49.14  210.90  4.291819      4.293075              NaN   
1091 2026-01-05  48.98  210.27  4.292977      4.293075              NaN   
1092 2026-01-06  49.13  210.77  4.290047      4.293075              NaN   
1093 2026-01-07  48.64  208.73  4.291324      4.293075              NaN   
1094 2026-01-08  49.74  213.42  4.290712      4.293075              NaN   
...         ...    ...     ...       ...           ...              ...   
1240 2026-08-10  53.71  230.39  4.289518      4.293075         4.292659   
1241 2026-08-11  53.62  229.95  4.288512      4.293075         4.292598   
1242 2026-08-12  53.82  231.10  4.293943      4.293075         4.292522   
1243 2026-08-13  54.30  233.09  4.292634      4.293075         4.292607   
1244 2026-08-14  54.35  233.28  4.292180      4.293075         4.292399   

      VDC/FSTA 20 dma    FSTA avg  FSTA 10 dma  FSTA 20 dma  avg diff  \
1090              NaN  210

In [40]:


prices_path = (
    notebook_directory.parent #.parent
    / "backtesting"
    / "backtests"
    / f"{output_filename}.csv"
)

df.to_csv(prices_path, index=False)